In [0]:
from sklearn.ensemble import IsolationForest

In [0]:
def ident_outliers_isolation_forest(dados, target, filtro = "filtro", coluna = "coluna", n_estimators=100, max_samples='auto', contamination='auto', max_features=1.0, bootstrap=False, n_jobs=None, random_state=None, verbose=0, warm_start=False):
  # https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/3749511783433397/command/3749511783433409
  '''
  dado = recebe um dataset completo com variáveis preditoras e target - o dataset necessita estar em formato Pandas.
  target = nome da coluna que há a variável target
  retorno = dataset com novas colunas agregadas, contendo 1 e -1. os registros que possuirem -1 em qualquer coluna de identificação são consideradas outliers.
  '''
  dados = dados.toPandas() #transforma os dados em pandas
  colunas = dados.columns #armazena as colunas em uma lista
  tg_list = dados[target].unique() #* verificar as classes distintas
  name_col = list(dados.drop(target, axis=1)) #* selecionar a coluna
  df_retorno = []
  var_outlier = []
  for values, tg in enumerate(tg_list):
    subset = dados[dados[target] == tg] # quebar o dataset por classe em outros df
    for variavel in name_col:
      clf = IsolationForest(n_estimators=n_estimators, max_samples=max_samples, contamination=contamination, max_features=max_features, bootstrap=bootstrap, n_jobs=n_jobs, random_state=random_state, verbose=verbose, warm_start=warm_start) #estimador
      X = np.array(subset[variavel]) #transforma as variaveis target em array
      X = np.reshape(X, (len(X), 1)) #dá um reshape nos dados para que sejam armazenados linhas(registros ao invez de coluna)
      clf.fit(X)
      predict = clf.predict(X)
      subset[f'{variavel}_outlier'] = predict #armazena a informação para uma nova coluna
      filtro_coluna = f'{variavel}_outlier' #filtra a coluna trabalhada no loop
      subset = subset[subset[filtro_coluna] > 0] #filtra os dados preditos (-1) da coluna trabalhada no loop
    df_retorno.append(subset) #junta as classes
  retorno = pd.concat(df_retorno) #armazena os dados numa variável
  retorno = retorno[colunas] # filtra as colunas do dataset de entrada
  retorno = spark.createDataFrame(retorno) #transformando no dataframe spark
  return retorno